In [1]:
import sqlite3
import joblib
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [2]:
conn = sqlite3.connect("../data/soccer.db")
scaler = StandardScaler()
ROLLING_WINDOW = 5
INITIAL_ELO = 1500
ELO_K = 20
HOME_ADVANTAGE = 100

In [3]:
query = """
SELECT
    m.*,
    c.name AS competition_name

FROM matches m

JOIN competitions c
ON m.competition_id = c.id
"""

matches_df = pd.read_sql(query, conn)
matches_df["result"] = matches_df.apply(
    lambda row:
        "H" if row["home_goals"] > row["away_goals"]
        else "A" if row["home_goals"] < row["away_goals"]
        else "D",
    axis=1
)
matches_df["total_goals"] = (
    matches_df["home_goals"] +
    matches_df["away_goals"]
)

In [4]:
completed_matches = matches_df[
    matches_df["status"].isin([
        "FT",
        "AET",
        "PEN"
    ])
].copy()
completed_matches = completed_matches.sort_values(
    by="date"
)

In [5]:
completed_matches = completed_matches.sort_values(by="date").copy()

completed_matches["result"] = completed_matches.apply(
    lambda row:
        "H" if row["home_goals"] > row["away_goals"]
        else "A" if row["home_goals"] < row["away_goals"]
        else "D",
    axis=1
)

completed_matches["home_win"] = (
    completed_matches["result"] == "H"
)

In [6]:
home_df = completed_matches[
    [
        "date",
        "competition_name",
        "season",
        "home_team_id",
        "home_goals",
        "away_goals"
    ]
].copy()

home_df.columns = [
    "date",
    "competition_name",
    "season",
    "team_id",
    "goals_scored",
    "goals_conceded"
]

away_df = completed_matches[
    [
        "date",
        "competition_name",
        "season",
        "away_team_id",
        "away_goals",
        "home_goals"
    ]
].copy()

away_df.columns = [
    "date",
    "competition_name",
    "season",
    "team_id",
    "goals_scored",
    "goals_conceded"
]

team_history = pd.concat([home_df, away_df])

team_history = team_history.sort_values(
    by=["team_id", "date"]
)

team_history["rolling_goals_scored"] = (
    team_history
    .groupby("team_id")["goals_scored"]
    .transform(
        lambda x: x.shift(1).rolling(ROLLING_WINDOW).mean()
    )
)

team_history["rolling_goals_conceded"] = (
    team_history
    .groupby("team_id")["goals_conceded"]
    .transform(
        lambda x: x.shift(1).rolling(ROLLING_WINDOW).mean()
    )
)

In [7]:
home_features = team_history.copy()

home_features = home_features.rename(
    columns={
        "team_id": "home_team_id",
        "rolling_goals_scored": "home_rolling_scored",
        "rolling_goals_conceded": "home_rolling_conceded"
    }
)

away_features = team_history.copy()

away_features = away_features.rename(
    columns={
        "team_id": "away_team_id",
        "rolling_goals_scored": "away_rolling_scored",
        "rolling_goals_conceded": "away_rolling_conceded"
    }
)

matches_features = completed_matches.merge(
    home_features[
        [
            "date",
            "home_team_id",
            "home_rolling_scored",
            "home_rolling_conceded"
        ]
    ],
    on=["date", "home_team_id"],
    how="left"
)

matches_features = matches_features.merge(
    away_features[
        [
            "date",
            "away_team_id",
            "away_rolling_scored",
            "away_rolling_conceded"
        ]
    ],
    on=["date", "away_team_id"],
    how="left"
)
matches_features["home_dnb"] = (
    matches_features["result"] != "A"
)
matches_features["away_dnb"] = (
    matches_features["result"] != "H"
)

In [8]:
elo_matches = completed_matches.sort_values(by="date").copy()

elo_ratings = {}
elo_home = []
elo_away = []

K = ELO_K

for _, match in elo_matches.iterrows():

    home_team = match["home_team_id"]
    away_team = match["away_team_id"]

    home_elo = elo_ratings.get(home_team, INITIAL_ELO)
    away_elo = elo_ratings.get(away_team, INITIAL_ELO)

    elo_home.append(home_elo)
    elo_away.append(away_elo)

    expected_home = 1 / (1 + 10 ** ((away_elo - home_elo) / 400))
    expected_away = 1 - expected_home

    if match["home_goals"] > match["away_goals"]:
        actual_home = 1
        actual_away = 0
    elif match["home_goals"] < match["away_goals"]:
        actual_home = 0
        actual_away = 1
    else:
        actual_home = 0.5
        actual_away = 0.5

    new_home_elo = home_elo + K * (actual_home - expected_home)
    new_away_elo = away_elo + K * (actual_away - expected_away)

    elo_ratings[home_team] = new_home_elo
    elo_ratings[away_team] = new_away_elo

elo_matches["home_elo"] = elo_home
elo_matches["away_elo"] = elo_away

matches_features = matches_features.merge(
    elo_matches[
        [
            "id",
            "home_elo",
            "away_elo"
        ]
    ],
    on="id",
    how="left"
)

In [9]:
context_query = """
SELECT
    *
FROM match_context
"""
context_df = pd.read_sql(
    context_query,
    conn
)
matches_features = matches_features.merge(

    context_df,

    left_on="id",

    right_on="match_id",

    how="left"
)

In [10]:
features = [

    # rolling
    "home_rolling_scored",
    "away_rolling_scored",

    "home_rolling_conceded",
    "away_rolling_conceded",

    # elo
    "home_elo",
    "away_elo",


    # coach
    "home_coach_tenure_days",
    "away_coach_tenure_days",

    # motivation
    "home_title_race",
    "away_title_race",

    "home_europe_race",
    "away_europe_race",

    "home_relegation_risk",
    "away_relegation_risk"
]

In [11]:
target = "home_dnb"
model_df = matches_features.dropna(
    subset=features + [target]
).copy()
train_df = model_df[
    model_df["season"] <= 2023
]

test_df = model_df[
    model_df["season"] >= 2024
]

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)
probabilities = model.predict_proba(
    X_test_scaled
)
home_dnb_proba = probabilities[:, 1]
test_results = test_df.copy()
test_results["prediction_proba"] = (
    home_dnb_proba
)
predictions = model.predict(X_test_scaled)

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

       False       0.58      0.28      0.38      1007
        True       0.73      0.90      0.80      2130

    accuracy                           0.70      3137
   macro avg       0.65      0.59      0.59      3137
weighted avg       0.68      0.70      0.67      3137



In [12]:
#Calibración
base_model = LogisticRegression(
    max_iter=1000
)
calibrated_model = CalibratedClassifierCV(

    base_model,

    method="sigmoid",

    cv=5
)
calibrated_model.fit(
    X_train_scaled,
    y_train
)
probabilities = calibrated_model.predict_proba(
    X_test_scaled
)
probabilities[:10]


array([[0.11250627, 0.88749373],
       [0.34262443, 0.65737557],
       [0.25455172, 0.74544828],
       [0.68978706, 0.31021294],
       [0.17707205, 0.82292795],
       [0.43055065, 0.56944935],
       [0.08772569, 0.91227431],
       [0.27809272, 0.72190728],
       [0.30991652, 0.69008348],
       [0.2892841 , 0.7107159 ]])

In [13]:
predictions = calibrated_model.predict(
    X_test_scaled
)
print(
    classification_report(
        y_test,
        predictions
    )
)

              precision    recall  f1-score   support

       False       0.58      0.27      0.37      1007
        True       0.72      0.90      0.80      2130

    accuracy                           0.70      3137
   macro avg       0.65      0.59      0.59      3137
weighted avg       0.68      0.70      0.67      3137



In [14]:
test_results = test_df.copy()
test_results["proba"] = (
    home_dnb_proba
)
test_results["prediction"] = (
    test_results["proba"] >= 0.75
)
confident = test_results[
    test_results["proba"] >= 0.75
]
accuracy_score(

    confident["home_dnb"],

    confident["prediction"]
)

0.827252419955324

In [15]:
xgb_model = XGBClassifier(

    n_estimators=200,

    max_depth=4,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42
)
xgb_model = XGBClassifier(

    n_estimators=200,

    max_depth=4,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42
)
xgb_model.fit(
    X_train,
    y_train
)
xgb_predictions = xgb_model.predict(
    X_test
)
print(
    classification_report(
        y_test,
        xgb_predictions
    )
)

              precision    recall  f1-score   support

       False       0.56      0.31      0.40      1007
        True       0.73      0.89      0.80      2130

    accuracy                           0.70      3137
   macro avg       0.65      0.60      0.60      3137
weighted avg       0.68      0.70      0.67      3137



In [16]:
result_mapping = {

    "H": 0,
    "D": 1,
    "A": 2
}

matches_features["result_encoded"] = (
    matches_features["result"]
    .map(result_mapping)
)

In [17]:
target = "result_encoded" #over_2_5 o home_win

model_df = matches_features.dropna(
    subset=features + [target]
).copy()

train_df = model_df[
    model_df["season"] <= 2023
]

test_df = model_df[
    model_df["season"] >= 2024
]

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000,class_weight="balanced")

model.fit(X_train_scaled, y_train)

predictions = model.predict(X_test_scaled)

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.60      0.56      0.58      1344
           1       0.28      0.25      0.26       786
           2       0.48      0.57      0.52      1007

    accuracy                           0.48      3137
   macro avg       0.45      0.46      0.45      3137
weighted avg       0.48      0.48      0.48      3137

